# Permutation on Validation set to inspect feature contribution

In [ ]:
import sys
import warnings

sys.path.append("../")
from src.data_utils import get_data, get_models
from src.config import SEED, BASE_PATH, DEVICE
from src.nn_models import load_nn_clf

import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

print(f"Using device: {DEVICE}")
print(f"Path: {BASE_PATH}")

In [ ]:
MODEL_DIR = BASE_PATH / "models"
# Data
OUTCOME_DICT = {
    "surg": get_data("outcome_surg"),
    "bleed": get_data("outcome_bleed"),
    "asp": get_data("outcome_asp"),
    "mort": get_data("outcome_mort"),
    "reop": get_data("outcome_reop"),
}
# Models
model_prefix_list = ["lr", "lgbm", "svc"]
##Can use any X df for input dimension, all = # of features
nn_in_dim = OUTCOME_DICT["surg"]["X_train"].shape[1]

MODEL_DICT = {}
for outcome in OUTCOME_DICT.keys():
    MODEL_DICT[outcome] = get_models(model_prefix_list, outcome)
    nn_dir = MODEL_DIR / outcome / "nn.pt"
    MODEL_DICT[outcome]["nn"] = load_nn_clf(
        data_path=nn_dir, in_dim=nn_in_dim, device=DEVICE
    )

In [ ]:
for outcome_name, outcome_data in OUTCOME_DICT.items():

    X_val = outcome_data['X_val']
    y_val = outcome_data['y_val']
    for model_name, model in MODEL_DICT[outcome_name].items():
        result = permutation_importance(
            estimator=model,
            X=X_val,
            y=y_val,
            n_repeats=30,
            random_state=SEED,
            n_jobs=-1,
            scoring="roc_auc",  # or any sklearn scorer string
        )
        importances_mean = result.importances_mean
        importances_std = result.importances_std
        feature_names = np.array(X_val.columns)

        idx = np.argsort(importances_mean)  # sort from low to high

        plt.figure(figsize=(20,  15))
        plt.barh(feature_names[idx], importances_mean[idx],
                xerr=importances_std[idx], align="center")
        plt.xlabel("Decrease in accuracy")
        plt.title(f"{outcome_name}-{model_name}")
        plt.tight_layout()
        plt.show()
        plt.close()